# 08e THINGS-EEG1 test P-1: a shared network against borrowing (prespecified, scored once)

**Kaggle:** GPU T4, Internet on. **Inputs:** the outputs of 08a and 08b.

Three shared networks (seeds 13 to 15) are trained on the 22 development people; each of the 24 reserved people is
then scored at k = 250 and 500 labelled images (draws 13 to 15) against five comparators. Endpoint: the mean over k
and 100 to 200 ms of the network minus each comparator, averaged per person over draws and seeds. Decision:
Bonferroni one-sided bounds over the five comparators with margin 0.0035 (`docs/proposals/eeg1_confirmation_protocol.md`).
Everything is fixed in the cell below and in `eval/eeg1_decision.py`.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '08e-confirm', 'gpu-confirm'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
SESSION_START = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run the confirmation on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
RUN_CONFIRMATION = True
APPROVED_MANIFEST = json.loads('{"protocol": "N188-source-pilot-draft-v1", "approval_reference": "Roles fixed by the assignment algorithm before any score was computed; approved for preparing the development people\'s inputs.", "assignment_approved": true, "channel_mapping_checked": true, "pilot_spending_approved": true, "assignment_seed": 13, "assignment_algorithm": "Sort canonical eligible IDs by SHA256 of \'XSUB EEG1 pilot assignment v1|seed={seed}|{person}\', tie-break by ID; first22 source, next24 confirmation; first4 source-ranked IDs pseudo-targets. Store each role list in canonical ID order.", "eligibility_policy": "Strict published-flag exclusion: sub-01, sub-06, sub-18 and sub-23 are excluded; the original score-blind 22/24 allocation is kept.", "primary_sources": ["sub-04", "sub-07", "sub-11", "sub-12", "sub-13", "sub-14", "sub-15", "sub-17", "sub-20", "sub-21", "sub-24", "sub-25", "sub-26", "sub-29", "sub-30", "sub-34", "sub-35", "sub-38", "sub-39", "sub-46", "sub-48", "sub-49"], "confirmation_targets": ["sub-02", "sub-03", "sub-05", "sub-08", "sub-09", "sub-10", "sub-16", "sub-19", "sub-22", "sub-27", "sub-28", "sub-31", "sub-32", "sub-33", "sub-36", "sub-37", "sub-40", "sub-41", "sub-42", "sub-43", "sub-44", "sub-45", "sub-47", "sub-50"], "pseudo_targets": ["sub-26", "sub-29", "sub-30", "sub-48"], "warning": "No outcome or score was used to set or change the roles.", "metadata_sha256": "0de3b9f46fe038b0428cbec9d69f347bde4865a127168bbf6fb14376197e33e0", "audit_sha256": "ce80fc1246f76ceee3f9ced9044ca6cac132cf2156a1b1bbc6ae7b3a187abb39", "channel_mapping_reference": "Header audit consistent (sampling rate, reference, channels, signal size)."}')
FROZEN = json.loads('{"delta": 0.003547826037718968, "feature_sha256": "61d36682db158fc622efe721c7759f70f52868a828aa9a66d395fdff7318ec1f", "k_subsets_sha256": "3f3ede66115c9fefd34e15d0b93a5fc10ef3f12ed32e251a3aab1a79732e01c1", "budgets": [250, 500], "draws": [13, 14, 15], "core_seeds": [13, 14, 15], "epochs": 30, "window_s": [0.1, 0.2], "alpha": 0.05, "min_targets": 20, "protocol": "P-1 frozen"}')

import numpy as np
import pandas as pd
import torch

from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.data import things_eeg2 as e2
from cross_subject_encoding.eval import eeg1_decision as dec
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.baselines import best_source, gu_by_group, window_groups
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.models.shared_core import MultiSubjectEncoder, latent, train_shared_images
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.eeg1_pilot import PilotClock, sha256_file, source_file, validate_manifest
from cross_subject_encoding.utils.preflight import preflight

report_path = C.METRICS_DIR / "08e_status.json"
report = {"status": "disabled", "frozen": FROZEN, "completed_targets": [],
          "failed_targets": {}}
report_path.write_text(json.dumps(report, indent=2))
ALPHAS = [10.0 ** exponent for exponent in range(-2, 9)]
DEVICE = "cuda"
INPUT_ROOT = Path("/kaggle/input")
N_TRAIN, N_TEST, N_FEATURES, N_FIT = e1.N_TRAIN, e1.N_TEST, 1000, 16540
CHANNELS = e1.CHANNELS
N_CHANNELS, N_TIMES, N_REPS = len(CHANNELS), 100, e1.N_TEST_REPS
MIN_HOST_BYTES = 12 * 2**30

In [ ]:
def run_confirmation():
    clock = PilotClock(minutes=630, reserve_minutes=20)
    clock.start = SESSION_START
    checkpoint = C.METRICS_DIR / "08e_checkpoints"
    checkpoint.mkdir(exist_ok=True)
    rows, curves, names = [], [], []
    access = []
    budgets, draws, seeds = FROZEN["budgets"], FROZEN["draws"], FROZEN["core_seeds"]
    try:
        validate_manifest(APPROVED_MANIFEST)
        sources, targets = APPROVED_MANIFEST["primary_sources"], APPROVED_MANIFEST["confirmation_targets"]
        report.update(status="preparing", sources=sources, targets=targets)
        report_path.write_text(json.dumps(report, indent=2))
        preflight("cross-subject-brain-encoding-08e-outputs",
                  inputs=["eeg1_paper_kpca1000.npz", "08b_readiness.json", "eeg1_image_metadata.npy", "eeg1_k_subsets.json"],
                  need_gpu=True, min_gpu_gb=8, expect_output_gb=0.6)
        memory = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
        if int(memory["MemAvailable"].split()[0]) * 1024 < MIN_HOST_BYTES:
            raise RuntimeError("Need 12 GiB available host memory")
        feature_path = C.find_artifact("eeg1_paper_kpca1000.npz")
        if sha256_file(feature_path) != FROZEN["feature_sha256"]:
            raise ValueError("Features differ from the frozen 08b file")
        subsets_path = C.find_artifact("eeg1_k_subsets.json")
        if sha256_file(subsets_path) != FROZEN["k_subsets_sha256"]:
            raise ValueError("k-subsets differ from the frozen 08a file")
        subsets = json.loads(subsets_path.read_text())
        metadata = np.load(C.find_artifact("eeg1_image_metadata.npy"), allow_pickle=True).item()
        image_order = [f"{c}/{f}" for c, f in zip(metadata["train_img_concepts"], metadata["train_img_files"])]
        image_order += [f"{c}/{f}" for c, f in zip(metadata["test_img_concepts"], metadata["test_img_files"])]
        with np.load(feature_path) as features:
            z_train, z_test = features["train"].astype(np.float64), features["test"].astype(np.float64)
            if features["image"].tolist() != image_order:
                raise ValueError("Feature image order differs from the converted EEG metadata")
        if z_train.shape != (N_TRAIN, N_FEATURES) or z_test.shape != (N_TEST, N_FEATURES):
            raise ValueError("Unexpected feature shape")
        expected_times = np.round(-0.2 + np.arange(N_TIMES) / 100, 3)

        def load(person, part):
            clock.require(f"load {person}/{part}")
            path = source_file(INPUT_ROOT, person, part, sources + targets)
            access.append(str(path))
            release = e2.load_part(path)
            expected = (N_TRAIN, 1, N_CHANNELS, N_TIMES) if part == "training" else (N_TEST, N_REPS, N_CHANNELS, N_TIMES)
            if release["data"].shape != expected or list(release["ch_names"]) != CHANNELS:
                raise ValueError(f"Wrong EEG shape/channel order: {person}/{part}")
            if not np.allclose(release["times"], expected_times) or not np.isfinite(release["data"]).all():
                raise ValueError(f"Bad time axis or nonfinite EEG: {person}/{part}")
            return release["data"]

        mean, scale = z_train.mean(0), z_train.std(0) + 1e-8
        x_train = torch.as_tensor((z_train - mean) / scale, dtype=torch.float32, device=DEVICE)
        x_test = torch.as_tensor((z_test - mean) / scale, dtype=torch.float32, device=DEVICE)
        query = np.concatenate([z_train, z_test])
        source_train = np.empty((len(sources), N_TRAIN, N_CHANNELS * N_TIMES), dtype=np.float32)
        source_test = np.empty((len(sources), N_TEST, N_CHANNELS * N_TIMES), dtype=np.float32)
        ys = {}
        for i, person in enumerate(sources):
            y_source = load(person, "training")[:, 0].reshape(N_TRAIN, -1).astype(np.float32)
            prediction = pl.ols_paper(z_train, y_source, query)
            source_train[i], source_test[i] = prediction[:N_TRAIN], prediction[N_TRAIN:]
            ys[person] = torch.as_tensor(y_source, device=DEVICE)
            del y_source, prediction
        report["status"] = "training"
        report_path.write_text(json.dumps(report, indent=2))
        cores, training = {}, {}
        for seed in seeds:
            clock.require(f"core seed {seed}")
            torch.manual_seed(seed)
            model = MultiSubjectEncoder(N_FEATURES, {p: N_CHANNELS * N_TIMES for p in sources}).to(DEVICE)
            permutation = np.random.default_rng(seed).permutation(N_TRAIN)
            hold, train = np.sort(permutation[-N_TRAIN // 10:]), np.sort(permutation[:-N_TRAIN // 10])
            training[seed] = train_shared_images(model, x_train, ys, train, hold, FROZEN["epochs"], 1e-3, 256, seed,
                                                 log=print, checkpoint_dir=checkpoint, tag=f"08e_core_seed{seed}")
            lat_train, lat_test = latent(model, x_train), latent(model, x_test)
            weight, bias = (tensor.cpu().numpy() for tensor in model.mean_readout())
            lat_mean, lat_scale = lat_train.mean(0), lat_train.std(0) + 1e-8
            cores[seed] = {"offset_train": (lat_train @ weight.T + bias).astype(np.float32),
                           "offset_test": lat_test @ weight.T + bias,
                           "both_train": np.hstack([(lat_train - lat_mean) / lat_scale, (z_train - mean) / scale]),
                           "both_test": np.hstack([(lat_test - lat_mean) / lat_scale, (z_test - mean) / scale])}
            del model
        del ys
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        report["training"] = {str(seed): result for seed, result in training.items()}
        groups = window_groups(expected_times, N_CHANNELS, C.EEG_WINDOW_MS)
        window = (expected_times >= FROZEN["window_s"][0]) & (expected_times < FROZEN["window_s"][1])
        report["status"] = "scoring"
        for person in targets:
            try:
                clock.require(f"target {person}")
                full = load(person, "training")[:, 0].reshape(N_TRAIN, -1).astype(np.float32)
                biological = load(person, "test").astype(np.float32)
                predictions = {"avg_sources|all|all": source_test.mean(0)}
                for draw in draws:
                    order = np.asarray(subsets["orders"][person][str(draw)])
                    for budget in budgets:
                        indices = order[:budget]
                        target = full[indices]
                        own = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, draw).fit(z_train[indices], target)
                        own_test = own.predict(z_test)
                        oof = np.empty_like(target)
                        for fold in np.array_split(np.random.default_rng(draw).permutation(budget), C.RIDGE_FOLDS):
                            keep = np.setdiff1d(np.arange(budget), fold)
                            oof[fold] = own.refit(z_train[indices[keep]], target[keep]).predict(z_train[indices[fold]])
                        output_map = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, draw).fit(oof, target)
                        borrowed = source_train[:, indices]
                        best = best_source(list(borrowed), target)
                        converter = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, draw).fit(borrowed[best], target)
                        tag = f"{budget}|{draw}"
                        predictions[f"own_ridge|{tag}"] = own_test
                        predictions[f"own_ridge_oc|{tag}"] = output_map.predict(own_test)
                        predictions[f"gu_window|{tag}"] = gu_by_group(borrowed, target, source_test, groups, "pooled")
                        predictions[f"linear_converter|{tag}"] = converter.predict(source_test[best])
                        for seed, core in cores.items():
                            residual = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, draw).fit(
                                core["both_train"][indices], target - core["offset_train"][indices])
                            predictions[f"prior_both|{tag}|s{seed}"] = core["offset_test"] + residual.predict(core["both_test"])
                for name, prediction in predictions.items():
                    if not np.isfinite(prediction).all():
                        raise ValueError(f"Nonfinite {name} prediction")
                    predictions[name] = prediction.reshape(N_TEST, N_CHANNELS, N_TIMES)
                scored = paper_correlation(predictions, biological, n_iter=100)
                for name, correlation in scored["correlation"].items():
                    method, budget, draw, *seed = name.split("|")
                    rows.append({"person": person, "method": method, "k": budget, "draw": draw,
                                 "core_seed": seed[0] if seed else "", "r_100_200ms": float(correlation[:, window].mean())})
                    names.append(f"{person}|{name}")
                    curves.append(correlation.astype(np.float32))
                report["completed_targets"].append(person)
            except TimeoutError:
                raise
            except Exception as error:  # protocol: a failed target is excluded and reported, never imputed
                report["failed_targets"][person] = f"{type(error).__name__}: {error}"
            pd.DataFrame(rows).to_csv(C.METRICS_DIR / "08e_scores.csv", index=False)
            np.savez_compressed(C.METRICS_DIR / "08e_curves.npz", names=np.array(names), curves=np.stack(curves) if curves else np.zeros(0),
                                times=expected_times)
            report_path.write_text(json.dumps(report, indent=2))
            print(person, "done" if person in report["completed_targets"] else report["failed_targets"].get(person))
        frame = pd.DataFrame(rows)
        frame = frame[frame["person"].isin(report["completed_targets"])]
        scores = {}
        for (person, method, budget), group in frame.groupby(["person", "method", "k"]):
            scores[(person, method, budget)] = group["r_100_200ms"].mean()  # draws (and core seeds) averaged
        expanded = {}
        for (person, method, budget), value in scores.items():
            for k in budgets:
                if budget == "all" or budget == str(k):
                    expanded[(person, method, k)] = value
        d = dec.endpoint(expanded, targets, budgets)
        decision = dec.decide(d, FROZEN["delta"], FROZEN["alpha"], FROZEN["min_targets"])
        decision["sign_flip_p"] = {c: dec.sign_flip_p(d[:, j]) for j, c in enumerate(dec.COMPARATORS)}
        seed_spread = {}
        for seed in seeds:
            per_seed = {key: value for key, value in expanded.items() if key[1] != "prior_both"}
            subset = frame[(frame["method"] != "prior_both") | (frame["core_seed"] == f"s{seed}")]
            for (person, budget), group in subset[subset["method"] == "prior_both"].groupby(["person", "k"]):
                per_seed[(person, "prior_both", int(budget))] = group["r_100_200ms"].mean()
            seed_spread[str(seed)] = np.nanmean(dec.endpoint(per_seed, targets, budgets), axis=0).tolist()
        decision["per_seed_mean_d"] = seed_spread
        decision["d"] = {person: row.tolist() for person, row in zip(targets, d)}
        (C.METRICS_DIR / "08e_decision.json").write_text(json.dumps(decision, indent=2))
        print(json.dumps({k: decision[k] for k in ("verdict", "n_targets", "n_excluded", "delta")}),
              json.dumps(decision["per_comparator"], indent=1))
        report.update(status="complete_confirmation", verdict=decision["verdict"])
    except TimeoutError as error:
        report.update(status="incomplete_time_budget", reason=str(error))
    except Exception as error:
        report.update(status="blocked_or_failed", reason=f"{type(error).__name__}: {error}")
        raise
    finally:
        report.update(eeg_files_opened=len(access))
        report_path.write_text(json.dumps(report, indent=2))
        print(json.dumps({k: v for k, v in report.items() if k != "frozen"}, indent=1)[:3000])


if RUN_CONFIRMATION:
    run_confirmation()
else:
    print("Confirmation disabled")

In [ ]:
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

export_to_dataset(slug="cross-subject-brain-encoding-08e-outputs", title="XSUB 08e confirmation outputs",
                  files=[report_path], include_dirs=[C.METRICS_DIR], message=report["status"])